# 01 · Datos: descarga, limpieza y exploración
**Responsable:** Persona A

Genera `data/processed/juegos.parquet`, que es la base de todos los demás notebooks.
La lógica de limpieza vive en `src/datos.py`; aquí solo se corre y se explora.

Decisiones (ver `src/config.py`):
- Se usa `games.csv` (el parquet de Hugging Face trae las etiquetas vacías). Su encabezado está corrido una columna y se corrige al leer.
- Solo juegos con **≥ 20 reseñas** (con menos, el % positivo es ruido).
- Objetivo: `bien_recibido = 1` si `pct_positivo > 0.80`.
- `Price` es el precio *con* el descuento del día de la foto → `precio_lista_usd = Price / (1 - Discount)`.
- Quetzales con tasa fija `TASA_GTQ_USD`; precio en oferta con `DESCUENTO_TIPICO` (plan B).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # para poder importar src/ desde notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config as C
from src.datos import cargar_juegos

df = cargar_juegos()
df.shape

In [ ]:
df.head()

## Variable objetivo

In [ ]:
print(df["bien_recibido"].value_counts(normalize=True))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["pct_positivo"], bins=50)
ax.axvline(C.UMBRAL_BIEN_RECIBIDO, color="k", ls="--", label="umbral 80 %")
ax.set(xlabel="% reseñas positivas", ylabel="juegos", title="Distribución del % positivo")
ax.legend()
fig.savefig(C.FIGURAS / "01_pct_positivo.png", dpi=120, bbox_inches="tight")

## Precio y descuento

In [ ]:
pagos = df[~df["es_gratis"]]
print(f"Gratis: {df['es_gratis'].mean():.1%}")
print(pagos["precio_lista_gtq"].describe())
print(f"Con descuento el día de la foto: {(df['descuento_actual'] > 0).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(pagos["precio_lista_gtq"].clip(upper=600), bins=60)
axes[0].set(xlabel="precio de lista (Q, recortado en 600)", ylabel="juegos")
axes[1].hist(df.loc[df["descuento_actual"] > 0, "descuento_actual"], bins=20)
axes[1].set(xlabel="descuento actual", ylabel="juegos")
fig.savefig(C.FIGURAS / "01_precio_descuento.png", dpi=120, bbox_inches="tight")

## Etiquetas, géneros y antigüedad

In [ ]:
etiquetas = df["etiquetas"].explode()
print("Etiquetas únicas:", etiquetas.nunique())
print("Juegos sin etiquetas:", (df["etiquetas"].map(len) == 0).sum())
etiquetas.value_counts().head(20)

In [ ]:
tasa = (df[["etiquetas", "bien_recibido"]].explode("etiquetas")
          .groupby("etiquetas")["bien_recibido"].agg(["mean", "size"])
          .query("size >= 500").sort_values("mean"))
pd.concat([tasa.head(8), tasa.tail(8)])

In [ ]:
por_anio = df.groupby("anio").agg(juegos=("appid", "size"), pct_bien=("bien_recibido", "mean"))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(por_anio.index, por_anio["juegos"])
ax2 = ax.twinx(); ax2.plot(por_anio.index, por_anio["pct_bien"], color="k", marker="o")
ax.set(xlabel="año de lanzamiento", ylabel="juegos"); ax2.set_ylabel("% bien recibidos")
fig.savefig(C.FIGURAS / "01_por_anio.png", dpi=120, bbox_inches="tight")

## Notas para el equipo
- Las columnas de listas (`etiquetas`, `generos`, ...) vuelven del parquet como `numpy.ndarray`; usar `list(x)` si hace falta.
- **No usar como predictores** las columnas de `C.COLUMNAS_PROHIBIDAS` (fuga de datos).